# TensorFlow简介

欢迎收看本周的编程作业！到目前为止，你一直使用 NumPy 来构建神经网络，但本周你将探索一个深度学习框架，它可以让你更轻松地构建神经网络。 TensorFlow、PaddlePaddle、Torch、Caffe、Keras 等机器学习框架可以显著加快机器学习开发速度。 TensorFlow 2.3 比其前身做出了重大改进，其中一些改进你将在这里遇到并实现！

在本作业结束时，你将能够在 TensorFlow 2.3 中执行以下操作：

* 使用 `tf.Variable` 修改变量的状态
* 解释变量和常量之间的区别
* 在 TensorFlow 数据集上训练神经网络

像 TensorFlow 这样的编程框架不仅可以减少编码时间，还可以执行优化来加快代码本身的速度。 

## 关于提交至 AutoGrader 的重要说明

向 AutoGrader 提交作业前，请确认满足以下要求：

1. 未在作业中添加 _额外的_  `print` 语句。
2. 未在作业中添加 _额外的_ 代码单元格。
3. 未修改任何函数的参数。
4. 未在计分练习中使用全局变量。除非明确要求使用全局变量，否则应使用局部变量。
5. 未修改无需改动的作业代码，例如创建 _额外的_ 变量。

如果违反上述要求，提交作业时可能出现 `Grader Error: Grader feedback not found` 或其他意外错误。在求助或调试作业之前，请先检查这些情况。如果不记得自己做过哪些改动，可以按照 [说明](https://www.coursera.org/learn/deep-neural-network/supplement/QWEnZ/h-ow-to-refresh-your-workspace) 获取一份新的作业。

## 目录
- [1- 依赖库](#1)
    - [1.1 - 检查TensorFlow版本](#1-1)
- [2 - 使用 GradientTape 进行基本优化](#2)
    - [2.1 - 线性函数](#2-1)
        - [练习 1 - linear_function](#ex-1)
    - [2.2 - 计算 sigmoid](#2-2)
        - [练习 2 - sigmoid](#ex-2)
    - [2.3 - 使用 one-hot 编码](#2-3)
        - [练习 3 - one_hot_matrix](#ex-3)
    - [2.4 - 初始化参数](#2-4)
        - [练习 4 - initialize_parameters](#ex-4)
- [3 - 在 TensorFlow 中构建你的第一个神经网络](#3)
    - [3.1 - 实现前向传播](#3-1)
        - [练习 5 - forward_propagation](#ex-5)
    - [3.2 计算总损失](#3-2)
        - [练习 6 - compute_total_loss](#ex-6)
    - [3.3 - 训练模型](#3-3)
- [4 - 参考文献](#4)

<a name='1'></a>
## 1 - 依赖库

In [ ]:
### v3.1

In [ ]:
import h5py
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.python.framework.ops import EagerTensor
from tensorflow.python.ops.resource_variable_ops import ResourceVariable
import time

<a name='1-1'></a>
### 1.1 - 检查TensorFlow版本 

你将为此作业使用 v2.3，以获得最大速度和效率。

In [ ]:
tf.__version__

<a name='2'></a>
## 2 - 使用 GradientTape 进行基本优化

TensorFlow 2 的优势在于简洁：你主要需要实现计算图中的前向传播，TensorFlow 会沿 `GradientTape` 记录的计算图反向计算导数。之后，只需指定代价函数和优化器即可。

编写TensorFlow程序时，主要使用和转换的对象是`tf.Tensor`。这些张量是 NumPy 数组 i.e 的 TensorFlow 等效项。给定数据类型的多维数组，还包含有关计算图的信息。

下面使用 `tf.Variable` 保存变量的状态。创建变量时，初始值会确定其形状和类型，后续通过赋值更新状态。此外，可以通过 `tf.Variable` 的 `dtype` 参数指定数据类型。若未指定，当初始值是 Tensor 时会保留其类型；否则，由 `convert_to_tensor` 推断类型。建议显式指定类型，以减少类型不匹配的问题。


在这里，你将调用在 HDF5 文件上创建的 TensorFlow 数据集，你可以使用它代替 NumPy 数组来存储数据集。你可以将其视为 TensorFlow 数据生成器！ 

你将使用手势数据集，该数据集由形状为 64x64x3 的图像组成。

In [ ]:
train_dataset = h5py.File('datasets/train_signs.h5', "r")
test_dataset = h5py.File('datasets/test_signs.h5', "r")

In [ ]:
x_train = tf.data.Dataset.from_tensor_slices(train_dataset['train_set_x'])
y_train = tf.data.Dataset.from_tensor_slices(train_dataset['train_set_y'])

x_test = tf.data.Dataset.from_tensor_slices(test_dataset['test_set_x'])
y_test = tf.data.Dataset.from_tensor_slices(test_dataset['test_set_y'])

In [ ]:
type(x_train)

由于 TensorFlow 数据集是生成器，因此你无法直接访问内容，除非你在 for 循环中迭代它们，或者使用 `iter` 显式创建 Python 迭代器并使用其内容
使用 `next` 的元素。此外，你还可以使用 `element_spec` 属性检查每个元素的 `shape` 和 `dtype`。

In [ ]:
print(x_train.element_spec)

In [ ]:
print(next(iter(x_train)))

你将在此作业中使用的数据集是手语数字的子集。它包含六个不同的类，代表从 0 到 5 的数字。

In [ ]:
unique_labels = set()
for element in y_train:
    unique_labels.add(element.numpy())
print(unique_labels)

你可以通过运行以下单元格来查看数据集中的一些图像。

In [ ]:
images_iter = iter(x_train)
labels_iter = iter(y_train)
plt.figure(figsize=(10, 10))
for i in range(25):
    ax = plt.subplot(5, 5, i + 1)
    plt.imshow(next(images_iter).numpy().astype("uint8"))
    plt.title(next(labels_iter).numpy().astype("uint8"))
    plt.axis("off")

TensorFlow 数据集和 NumPy 数组之间还有一个额外的区别：如果你需要转换一个，你可以调用 `map` 方法来将作为参数传递的函数应用于每个元素。

In [ ]:
def normalize(image):
    """
    将图像转换为形状为 (64 * 64 * 3, ) 的张量
    和归一化其组件。
    
    参数
    图像 - 张量。
    
    返回： 
    result -- 变换后的张量 
    """
    image = tf.cast(image, tf.float32) / 255.0
    image = tf.reshape(image, [-1,])
    return image

In [ ]:
new_train = x_train.map(normalize)
new_test = x_test.map(normalize)

In [ ]:
new_train.element_spec

In [ ]:
print(next(iter(new_train)))

<a name='2-1'></a>
### 2.1 - 线性函数

让我们通过计算以下方程开始此编程练习：$Y = WX + b$，其中 $W$ 和 $X$ 是随机矩阵，b 是随机向量。 

<a name='ex-1'></a>
### 练习 1 - linear_function

计算 $WX + b$，其中 $W, X$ 和 $b$ 是从随机正态分布中抽取的。 W 的形状为 (4, 3)，X 的形状为 (3,1)，b 的形状为 (4,1)。作为示例，以下是如何定义形状为 (3,1) 的常量 X：
```python
X = tf.constant(np.random.randn(3,1), name = "X")

```
 请注意，`tf.constant` 和 `tf.Variable` 之间的区别在于，你可以修改 `tf.Variable` 的状态，但不能更改 `tf.constant` 的状态。

你可能会发现以下函数有帮助： 
- tf.matmul(..., ...) 进行矩阵乘法
- tf.add(..., ...) 进行加法
- np.random.randn(...) 随机初始化

In [ ]:
# GRADED FUNCTION: linear_function

def linear_function():
    """
    实现线性函数： 
            将 X 初始化为形状为 (3,1) 的随机张量
            将 W 初始化为形状为 (4,3) 的随机张量
            将 b 初始化为形状为 (4,1) 的随机张量
    返回： 
    result -- Y = WX + b 
    """

    np.random.seed(1)
    
    """
    注意，为确保生成的 "random" 数字与预期结果匹配，
    请按照下面起始代码中给出的顺序创建变量。
    （请勿重新排列顺序）。
    """
    # （约4行）
    # X = ...
    # W = ...
    # b = ...
    # Y = ...
    # YOUR CODE STARTS HERE
    
    
    # YOUR CODE ENDS HERE
    return Y

In [ ]:
result = linear_function()
print(result)

assert type(result) == EagerTensor, "Use the TensorFlow API"
assert np.allclose(result, [[-2.15657382], [ 2.95891446], [-1.08926781], [-0.84538042]]), "Error"
print("\033[92mAll test passed")


**预期输出**： 

```
result = 
[[-2.15657382]
 [ 2.95891446]
 [-1.08926781]
 [-0.84538042]]
```

<a name='2-2'></a>
### 2.2 - 计算 sigmoid 
太棒了！你刚刚实现了一个线性函数。 TensorFlow提供了多种常用的神经网络函数，如`tf.sigmoid`和`tf.softmax`。

在本练习中，计算 z 的 sigmoid。 

在本练习中，你将： 使用 `tf.cast` 将张量转换为 `float32`，然后使用 `tf.keras.activations.sigmoid` 计算 sigmoid。 

<a name='ex-2'></a>
### 练习 2 - sigmoid

实现下面的 sigmoid 函数。你应该使用以下内容： 

- `tf.cast("...", tf.float32)`
- `tf.keras.activations.sigmoid("...")`

In [ ]:
# GRADED FUNCTION: sigmoid

def sigmoid(z):
    
    """
    计算 z 的 sigmoid
    
    参数：
    z -- 输入值，标量或向量
    
    返回： 
    a -- (tf.float32) z 的 sigmoid
    """
    # tf.keras.activations.sigmoid 需要 float16、float32、float64、complex64 或complex128。
    
    # （约2行）
    # z = ...
    # a = ...
    # YOUR CODE STARTS HERE
    
    
    # YOUR CODE ENDS HERE
    return a


In [ ]:
result = sigmoid(-1)
print ("type: " + str(type(result)))
print ("dtype: " + str(result.dtype))
print ("sigmoid(-1) = " + str(result))
print ("sigmoid(0) = " + str(sigmoid(0.0)))
print ("sigmoid(12) = " + str(sigmoid(12)))

def sigmoid_test(target):
    result = target(0)
    assert(type(result) == EagerTensor)
    assert (result.dtype == tf.float32)
    assert sigmoid(0) == 0.5, "Error"
    assert sigmoid(-1) == 0.26894143, "Error"
    assert sigmoid(12) == 0.9999939, "Error"

    print("\033[92mAll test passed")

sigmoid_test(sigmoid)

**预期输出**： 
<table>
<tr> 
<td>
类型
</td>
<td>
类“tensorflow.python.framework.ops.EagerTensor”
</td>
</tr><tr> 
<td>
数据类型
</td>
<td>
“数据类型：‘float32’
</td>
</tr>
<tr> 
<td>
Sigmoid(-1)
</td>
<td>
0.2689414
</td>
</tr>
<tr> 
<td>
Sigmoid(0)
</td>
<td>
0.5
</td>
</tr>
<tr> 
<td>
Sigmoid(12)
</td>
<td>
0.999994
</td>
</tr> 

</table> 

<a name='2-3'></a>
### 2.3 - 使用 one-hot 编码

在深度学习中，很多时候你会得到一个 $Y$ 向量，其数字范围从 $0$ 到 $C-1$，其中 $C$ 是类的数量。例如，如果 $C$ 是 4，那么你可能有以下 y 向量，你需要像这样进行转换：


<img src="images/onehot.png" style="width:600px;height:150px;">

这称为“one-hot”编码，因为在转换后的表示中，每列中只有一个元素是“热”的（即设置为 1）。要在 NumPy 中执行此转换，你可能需要编写几行代码。在TensorFlow中，可以使用一行代码： 

- [tf.one_hot(labels, depth, axis=0)](https://www.tensorflow.org/api_docs/python/tf/one_hot)

`axis=0` 表示新轴在维度 0 处创建

<a name='ex-3'></a>
### 练习 3 - one_hot_matrix

实现下面的函数来获取一个标签和类总数 $C$，并以一维张量（数组）形式返回 one-hot 编码（你可以查看下面的预期输出以了解想法）。使用 `tf.one_hot()` 来执行此操作，并使用 `tf.reshape()` 来重塑你的 one-hot 张量！ 

- `tf.reshape(tensor, shape)`

In [ ]:
# GRADED FUNCTION: one_hot_matrix
def one_hot_matrix(label, C=6):
    """
    计算单个标签的 one-hot 编码
    
    参数：
        label --  (int) 分类标签
        C --  (int) 标签可以采用的不同类别的数量
    
    返回：
         one_hot -- tf.Tensor 编码为 one-hot 的一维张量（数组）。
    """
    # （约1行）
    # one_hot = None(None(None, None, None), shape=[C, ]) 
    # YOUR CODE STARTS HERE
    
    
    # YOUR CODE ENDS HERE
    return one_hot

In [ ]:
def one_hot_matrix_test(target):
    label = tf.constant(1)
    C = 4
    result = target(label, C)
    print("Test 1:",result)
    assert result.shape[0] == C, "Use the parameter C"
    assert np.allclose(result, [0., 1. ,0., 0.] ), "Wrong output. Use tf.one_hot"
    label_2 = [2]
    C = 5
    result = target(label_2, C)
    print("Test 2:", result)
    assert result.shape[0] == C, "Use the parameter C"
    assert np.allclose(result, [0., 0. ,1., 0., 0.] ), "Wrong output. Use tf.reshape as instructed"
    
    print("\033[92mAll test passed")

one_hot_matrix_test(one_hot_matrix)

**预期输出**
```
Test 1: tf.Tensor([0. 1. 0. 0.], shape=(4,), dtype=float32)
Test 2: tf.Tensor([0. 0. 1. 0. 0.], shape=(5,), dtype=float32)
```

In [ ]:
new_y_test = y_test.map(one_hot_matrix)
new_y_train = y_train.map(one_hot_matrix)

In [ ]:
print(next(iter(new_y_test)))

<a name='2-4'></a>
### 2.4 - 初始化参数 

接下来，使用 Glorot 初始化器初始化一个数值向量。调用 `tf.keras.initializers.GlorotNormal`，从以 0 为中心、标准差参数为 `stddev = sqrt(2 / (fan_in + fan_out))` 的截断正态分布中采样。其中，`fan_in` 和 `fan_out` 分别表示权重张量对应的输入单元数和输出单元数。

要使用零或一进行初始化，你可以使用 `tf.zeros()` 或 `tf.ones()` 代替。 

<a name='ex-4'></a>
### 练习 4 - initialize_parameters

实现下面的函数以使用 GlorotNormal 初始值设定项获取形状并返回数字数组。 

 - `tf.keras.initializers.GlorotNormal(seed=1)`
 - `tf.Variable(initializer(shape=())`

In [ ]:
# GRADED FUNCTION: initialize_parameters

def initialize_parameters():
    """
    初始化参数以使用 TensorFlow 构建神经网络。形状是：
                        W1 : [25, 12288]
                        b1 : [25, 1]
                        W2 : [12, 25]
                        b2 : [12, 1]
                        W3 : [6, 12]
                        b3 : [6, 1]
    
    返回：
    parameters -- 包含W1，b1，W2，b2，W3，b3的张量字典
    """
                                
    initializer = tf.keras.initializers.GlorotNormal(seed=1)   
    #（约6行代码）
    # W1 = ...
    # b1 = ...
    # W2 = ...
    # b2 = ...
    # W3 = ...
    # b3 = ...
    # YOUR CODE STARTS HERE
    
    
    # YOUR CODE ENDS HERE

    parameters = {"W1": W1,
                  "b1": b1,
                  "W2": W2,
                  "b2": b2,
                  "W3": W3,
                  "b3": b3}
    
    return parameters

In [ ]:
def initialize_parameters_test(target):
    parameters = target()

    values = {"W1": (25, 12288),
              "b1": (25, 1),
              "W2": (12, 25),
              "b2": (12, 1),
              "W3": (6, 12),
              "b3": (6, 1)}

    for key in parameters:
        print(f"{key} shape: {tuple(parameters[key].shape)}")
        assert type(parameters[key]) == ResourceVariable, "All parameter must be created using tf.Variable"
        assert tuple(parameters[key].shape) == values[key], f"{key}: wrong shape"
        assert np.abs(np.mean(parameters[key].numpy())) < 0.5,  f"{key}: Use the GlorotNormal initializer"
        assert np.std(parameters[key].numpy()) > 0 and np.std(parameters[key].numpy()) < 1, f"{key}: Use the GlorotNormal initializer"

    print("\033[92mAll test passed")
    
initialize_parameters_test(initialize_parameters)

**预期输出**
```
W1 shape: (25, 12288)
b1 shape: (25, 1)
W2 shape: (12, 25)
b2 shape: (12, 1)
W3 shape: (6, 12)
b3 shape: (6, 1)
```

In [ ]:
parameters = initialize_parameters()

<a name='3'></a>
## 3 - 在 TensorFlow 中构建你的第一个神经网络

在作业的这一部分中，你将使用 TensorFlow 构建一个神经网络。请记住，实现 TensorFlow 模型分为两个部分：

- 实现前向传播
- 检索梯度并训练模型

让我们开始吧！

<a name='3-1'></a>
### 3.1 - 实现前向传播 

TensorFlow 的一大优势在于，你只需要实现前向传播函数，它就会自动跟踪你执行的操作来计算反向传播。  


<a name='ex-5'></a>
### 练习 5 - forward_propagation

实现 `forward_propagation` 函数。

**注意** 仅使用 TF API。 

- tf.math.add
- tf.linalg.matmul
- tf.keras.activations.relu

你不会在这里应用“softmax”。你将在下面的 `Exercise 6` 中看到 TensorFlow 如何在内部完成它的计算。

In [ ]:
# GRADED FUNCTION: forward_propagation

def forward_propagation(X, parameters):
    """
    为模型实现前向传播：LINEAR -> ReLU -> LINEAR -> ReLU -> LINEAR
    
    参数：
    X -- 输入数据集占位符，形状（输入大小，示例数量）
    parameters -- 包含参数的Python字典 "W1"、"b1"、"W2"、"b2"、"W3"、"b3"
                  形状在 initialize_parameters 中给出

    返回：
    Z3 -- 最后一个LINEAR单元的输出
    """
    
    # 从字典"parameters"中检索参数 
    W1 = parameters['W1']
    b1 = parameters['b1']
    W2 = parameters['W2']
    b2 = parameters['b2']
    W3 = parameters['W3']
    b3 = parameters['b3']
    
    #（约 5 行）# NumPy 等效项（NumPy 不使用。使用 TF API）：
    # Z1 = ...                           # Z1 = np.dot(W1, X) + b1
    # A1 = ...                           # A1 = relu(Z1)
    # Z2 = ...                           # Z2 = np.dot(W2, A1) + b2
    # A2 = ...                           # A2 = relu(Z2)
    # Z3 = ...                           # Z3 = np.dot(W3, A2) + b3
    # YOUR CODE STARTS HERE
    
    
    # YOUR CODE ENDS HERE
    
    return Z3

In [ ]:
def forward_propagation_test(target, examples):
    minibatches = examples.batch(2)
    parametersk = initialize_parameters()
    W1 = parametersk['W1']
    b1 = parametersk['b1']
    W2 = parametersk['W2']
    b2 = parametersk['b2']
    W3 = parametersk['W3']
    b3 = parametersk['b3']
    index = 0
    minibatch = list(minibatches)[0]
    with tf.GradientTape() as tape:
        forward_pass = target(tf.transpose(minibatch), parametersk)
        print(forward_pass)
        fake_cost = tf.reduce_mean(forward_pass - np.ones((6,2)))

        assert type(forward_pass) == EagerTensor, "Your output is not a tensor"
        assert forward_pass.shape == (6, 2), "Last layer must use W3 and b3"
        assert np.allclose(forward_pass, 
                           [[-0.13430887,  0.14086473],
                            [ 0.21588647, -0.02582335],
                            [ 0.7059658,   0.6484556 ],
                            [-1.1260961,  -0.9329492 ],
                            [-0.20181894, -0.3382722 ],
                            [ 0.9558965,   0.94167566]]), "Output does not match"
    index = index + 1
    trainable_variables = [W1, b1, W2, b2, W3, b3]
    grads = tape.gradient(fake_cost, trainable_variables)
    assert not(None in grads), "Wrong gradients. It could be due to the use of tf.Variable whithin forward_propagation"
    print("\033[92mAll test passed")

forward_propagation_test(forward_propagation, new_train)

**预期输出**
```
tf.Tensor(
[[-0.13430887  0.14086473]
 [ 0.21588647 -0.02582335]
 [ 0.7059658   0.6484556 ]
 [-1.1260961  -0.9329492 ]
 [-0.20181894 -0.3382722 ]
 [ 0.9558965   0.94167566]], shape=(6, 2), dtype=float32)
```

<a name='3-2'></a>
### 3.2 计算总损失

现在你所要做的就是定义你要使用的损失函数。对于本例，由于我们有一个包含 6 个标签的分类问题，因此分类交叉熵将起作用！

此前，计算代价时会先对全部样本（即所有 mini-batch）的损失求和，再除以样本总数。这里将分两步完成这一过程。

在步骤 1 中，`compute_total_loss` 函数将仅负责对一个 mini-batch 样本的损失求和。然后，当你训练模型（在第 3.3 节中）时，每个 mini-batch 都会调用一次 `compute_total_loss` 函数，步骤 2 将通过累加每个 mini-batch 的总和来完成，然后除以样本总数以获得最终代价。

在步骤1中计算“总损失”而不是“平均损失”可以确保最终代价一致。例如，如果 mini-batch 大小为 4，但整个批次中只有 5 个样本，则最后一个 mini-batch 将只有 1 个样本。考虑到 5 个样本，损失分别为 [0, 1, 2, 3, 4]，我们知道最终代价应该是它们的平均值，即 2。采用“总损失”方法将得到相同的答案。然而，“平均损失”方法首先会得到两个mini-batch的1.5和4，然后平均后最终得到2.75，这与期望的结果2不同。因此，这里采用“总损失”方法。 

<a name='ex-6'></a>
### 练习 6 - compute_total_loss

执行下面的总损失函数。你将使用它来计算一批样本的总损失。通过这个方便的函数，你可以将多个批次的损失相加，并将总和除以样本总数以获得代价。 
- 值得注意的是，[tf.keras.losses.categorical_crossentropy](https://www.tensorflow.org/api_docs/python/tf/keras/losses/categorical_crossentropy) 的“`y_pred`”和“`y_true`”输入预计具有形状（示例数量，num_classes）。 

- `tf.reduce_sum` 对示例进行求和。

- 你跳过了在 `Exercise 5` 中应用“softmax”，现在将由 `tf.keras.losses.categorical_crossentropy` 通过设置其参数 `from_logits=True` 来处理（你可以在社区中阅读我们的一位导师 [这里](https://community.deeplearning.ai/t/week-3-assignment-compute-total-loss-try-to-set-from-logits-false/243049/2?u=paulinpaloalto) 的回复，了解其背后的数学推理。如果你不是社区的一部分你已经可以通过 [这里](https://www.coursera.org/learn/deep-neural-network/ungradedLti/ZE1VR/important-have-questions-issues-or-ideas-join-our-community) 来完成此操作。）

In [ ]:
# GRADED FUNCTION: compute_total_loss 

def compute_total_loss(logits, labels):
    """
    计算总损失
    
    参数：
    logits -- 前向传播的输出（最后一个线性单元的输出），形状为 (6, num_examples)
    labels -- "true" 标签向量，与 Z3 形状相同
    
    返回：
    total_loss - 总损失值的张量
    """
    
    #（1行代码）
    # 记得设置`from_logits=True`
    # total_loss = ...
    # YOUR CODE STARTS HERE
    
    
    # YOUR CODE ENDS HERE
    return total_loss

In [ ]:
def compute_total_loss_test(target, Y):
    pred = tf.constant([[ 2.4048107,   5.0334096 ],
             [-0.7921977,  -4.1523376 ],
             [ 0.9447198,  -0.46802214],
             [ 1.158121,    3.9810789 ],
             [ 4.768706,    2.3220146 ],
             [ 6.1481323,   3.909829  ]])
    minibatches = Y.batch(2)
    for minibatch in minibatches:
        result = target(pred, tf.transpose(minibatch))
        break
    
    print("Test 1: ", result)
    assert(type(result) == EagerTensor), "Use the TensorFlow API"
    assert (np.abs(result - (0.50722074 + 1.1133534) / 2.0) < 1e-7), "Test 1 does not match. Did you get the reduce sum of your loss functions?"
    
    ### 测试 2
    labels = tf.constant([[1., 0., 0.], [0., 1., 0.], [0., 0., 1.]])
    logits = tf.constant([[1., 0., 0.], [1., 0., 0.], [1., 0., 0.]])
    
    result = compute_total_loss(logits, labels)
    print("Test 2: ", result)
    assert np.allclose(result, 3.295837 ), "Test 2 does not match."
    
    print("\033[92mAll test passed")

compute_total_loss_test(compute_total_loss, new_y_train )

**预期输出**
```
Test 1:  tf.Tensor(0.810287, shape=(), dtype=float32)
Test 2:  tf.Tensor(3.295837, shape=(), dtype=float32)
```

**注意：** 当使用损失总和进行梯度计算时，重要的是随着 mini-batch 大小的增加而减小学习率。这可以确保你不会朝着最小值迈出大步。

<a name='3-3'></a>
### 3.3 - 训练模型

让我们来谈谈优化器。你将在一行中指定优化器的类型，在本例中为 `tf.keras.optimizers.Adam`（尽管你可以使用其他优化器，例如 SGD），然后在训练循环中调用它。 

注意 `tape.gradient` 函数：这允许你检索 `GradientTape` 块内自动微分记录的操作。然后，调用优化器方法 `apply_gradients`，将优化器的更新规则应用于每个可训练参数。在本作业的末尾，你将找到一些更详细地解释这一点的文档，但现在，一个简单的解释就足够了。 ;) 


在这里你应该注意批量训练过程中添加的一个重要的额外步骤： 

- `tf.Data.dataset = dataset.prefetch(8)` 

这样做是为了防止从磁盘读取时可能出现的内存瓶颈。 `prefetch()` 保留一些数据并在需要时随时准备。它通过从输入数据创建源数据集、应用转换来预处理数据、然后一次迭代数据集指定数量的元素来实现此目的。这是可行的，因为迭代是流式的，因此数据不需要适合内存。 

In [ ]:
def model(X_train, Y_train, X_test, Y_test, learning_rate = 0.0001,
          num_epochs = 1500, minibatch_size = 32, print_cost = True):
    """
    实现三层TensorFlow神经网络：LINEAR->ReLU->LINEAR->ReLU->LINEAR->softmax。
    
    参数：
    X_train -- 训练集，形状（输入大小 = 12288，训练样本的数量 = 1080）
    Y_train -- 测试集，形状（输出大小 = 6，训练样本的数量 = 1080）
    X_test -- 训练集，形状（输入大小 = 12288，训练样本的数量 = 120）
    Y_test -- 测试集，形状（输出大小 = 6，测试示例数量 = 120）
    learning_rate -- 优化的学习率
    num_epochs -- 优化循环的epoch数
    minibatch_size -- mini-batch 的大小
    print_cost -- 正确打印每10个epoch的代价
    
    返回：
    parameters -- 模型学习到的参数。然后可以使用它们进行预测。
    """
    
    costs = []                                        # 跟踪代价
    train_acc = []
    test_acc = []
    
    # 初始化你的参数
    #（1 行）
    parameters = initialize_parameters()

    W1 = parameters['W1']
    b1 = parameters['b1']
    W2 = parameters['W2']
    b2 = parameters['b2']
    W3 = parameters['W3']
    b3 = parameters['b3']

    optimizer = tf.keras.optimizers.Adam(learning_rate)
    
    # CategoricalAccuracy 将跟踪此多类问题的准确性
    test_accuracy = tf.keras.metrics.CategoricalAccuracy()
    train_accuracy = tf.keras.metrics.CategoricalAccuracy()
    
    dataset = tf.data.Dataset.zip((X_train, Y_train))
    test_dataset = tf.data.Dataset.zip((X_test, Y_test))
    
    # 我们可以使用基数方法获取数据集的元素数量
    m = dataset.cardinality().numpy()
    
    minibatches = dataset.batch(minibatch_size).prefetch(8)
    test_minibatches = test_dataset.batch(minibatch_size).prefetch(8)
    #X_train = X_train.batch(minibatch_size, drop_remainder=True).prefetch(8)# <<< 额外步骤    
    #Y_train = Y_train.batch(minibatch_size, drop_remainder=True).prefetch(8) # 加载内存更快 

    # 进行训练循环
    for epoch in range(num_epochs):

        epoch_total_loss = 0.
        
        #我们需要重置对象以从 0 开始测量每个epoch的准确率
        train_accuracy.reset_states()
        
        for (minibatch_X, minibatch_Y) in minibatches:
            
            with tf.GradientTape() as tape:
                # 1.预测
                Z3 = forward_propagation(tf.transpose(minibatch_X), parameters)

                # 2. 损失
                minibatch_total_loss = compute_total_loss(Z3, tf.transpose(minibatch_Y))

            # 我们累积所有批次的准确率
            train_accuracy.update_state(minibatch_Y, tf.transpose(Z3))
            
            trainable_variables = [W1, b1, W2, b2, W3, b3]
            grads = tape.gradient(minibatch_total_loss, trainable_variables)
            optimizer.apply_gradients(zip(grads, trainable_variables))
            epoch_total_loss += minibatch_total_loss
        
        # 我们将 epoch 总损失除以样本数量
        epoch_total_loss /= m

        # 每 10 个 epoch 打印一次代价
        if print_cost == True and epoch % 10 == 0:
            print ("Cost after epoch %i: %f" % (epoch, epoch_total_loss))
            print("Train accuracy:", train_accuracy.result())
            
            # 我们每 10 个 epoch 评估一次测试集以避免计算开销
            for (minibatch_X, minibatch_Y) in test_minibatches:
                Z3 = forward_propagation(tf.transpose(minibatch_X), parameters)
                test_accuracy.update_state(minibatch_Y, tf.transpose(Z3))
            print("Test_accuracy:", test_accuracy.result())

            costs.append(epoch_total_loss)
            train_acc.append(train_accuracy.result())
            test_acc.append(test_accuracy.result())
            test_accuracy.reset_states()


    return parameters, costs, train_acc, test_acc

In [ ]:
parameters, costs, train_acc, test_acc = model(new_train, new_y_train, new_test, new_y_test, num_epochs=100)

**预期输出**

```
Cost after epoch 0: 1.830244
Train accuracy: tf.Tensor(0.17037037, shape=(), dtype=float32)
Test_accuracy: tf.Tensor(0.2, shape=(), dtype=float32)
Cost after epoch 10: 1.552390
Train accuracy: tf.Tensor(0.35925925, shape=(), dtype=float32)
Test_accuracy: tf.Tensor(0.30833334, shape=(), dtype=float32)
...
```
 你得到的数字可能会有所不同，只需检查你的损失是否在下降而你的准确率是否在上升！

In [ ]:
# 绘制代价图
plt.plot(np.squeeze(costs))
plt.ylabel('cost')
plt.xlabel('iterations (per fives)')
plt.title("Learning rate =" + str(0.0001))
plt.show()


In [ ]:
# 绘制列车精度图
plt.plot(np.squeeze(train_acc))
plt.ylabel('Train Accuracy')
plt.xlabel('iterations (per fives)')
plt.title("Learning rate =" + str(0.0001))
# 绘制测试精度图
plt.plot(np.squeeze(test_acc))
plt.ylabel('Test Accuracy')
plt.xlabel('iterations (per fives)')
plt.title("Learning rate =" + str(0.0001))
plt.show()


**恭喜**！你已经读完本作业以及本周材料的结尾。在 TensorFlow 2.3 中构建神经网络的惊人工作！ 

以下是你刚刚取得的所有成就的快速回顾：

- 使用 `tf.Variable` 修改变量
- 在 TensorFlow 数据集上训练神经网络

你现在可以利用 TensorFlow 的力量更快地创造很酷的东西。好的！ 

<a name='4'></a>
## 4 - 参考文献 

在这个作业中，向你介绍了`tf.GradientTape`，它记录了微分操作。以下是一些资源，可帮助你更深入地了解其用途和原因： 

梯度及自动微分简介： 
https://www.tensorflow.org/guide/autodiff 

GradientTape 文档：
https://www.tensorflow.org/api_docs/python/tf/GradientTape